# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gulzaibsharif-coder/flyrank-ml-capstone/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Action-ranking approach

The action queue prioritizes content items for human review using the validated refresh/opportunity signal developed in the previous modeling work.

Each item receives a priority score, rank, reason code, and recommended action. Reason codes translate observed search-performance signals into language that a content reviewer can understand.

The main reason codes are:

- `DECAY_RISK`: observed evidence of declining search performance.
- `RANKING_SLIP`: the page shows a weaker average search position relative to the review criteria.
- `HIGH_IMPRESSIONS_LOW_CTR`: the page receives search impressions but has relatively weak click-through performance.
- `CONTENT_OPPORTUNITY`: the page has a meaningful search footprint but signals suggest that additional review may be useful.
- `NO_STRONG_SIGNAL`: no single signal is strong enough to justify a more specific reason code.

The queue is decision-support rather than an automatic publishing system. A high score means that an item may be worth reviewing earlier; it does not prove that refreshing the item will improve search performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
# SECTION 1 — Build a ranked content-action queue

from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("/content/flyrank-ml-capstone/data/raw/content_refresh_anonymized.csv")

# If the repository is not mounted at /content/flyrank-ml-capstone,
# use the local path below.
if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Columns:", len(df.columns))

# Basic validation
required_cols = [
    "content_id",
    "client_id",
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
]

missing_required = [c for c in required_cols if c not in df.columns]

if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

# Work on a copy
queue = df.copy()

# IMPORTANT:
# avg_position == 0 means no position data, not rank zero.
queue["has_position_data"] = queue["avg_position"].gt(0)

# Normalize the useful signals for ranking.
# These are descriptive prioritization signals, not causal estimates.

def percentile_rank(s):
    return s.rank(pct=True).fillna(0)

queue["impression_signal"] = percentile_rank(
    queue["impressions"].clip(lower=0)
)

queue["ctr_signal"] = percentile_rank(
    queue["ctr"].clip(lower=0)
)

# Lower average position is better in search.
# We therefore reverse the percentile ranking.
valid_position = queue.loc[queue["has_position_data"], "avg_position"]

queue["position_signal"] = 0.0

if len(valid_position) > 0:
    queue.loc[queue["has_position_data"], "position_signal"] = (
        1 - percentile_rank(valid_position)
    )

# If the validated model probability exists from earlier cells,
# use it. Otherwise create a transparent fallback priority score
# from observed signals.
if "model_score" in queue.columns:
    queue["priority_score"] = queue["model_score"]
elif "predicted_probability" in queue.columns:
    queue["priority_score"] = queue["predicted_probability"]
else:
    # Conservative decision-support score.
    # It is explicitly a prioritization score, not a model probability.
    queue["priority_score"] = (
        0.40 * queue["impression_signal"]
        + 0.30 * (1 - queue["ctr_signal"])
        + 0.30 * queue["position_signal"]
    )

# Reason-code logic
def assign_reason(row):
    if "is_declining_label" in queue.columns and row.get("is_declining_label", 0) == 1:
        return "DECAY_RISK"

    if row["has_position_data"] and row["avg_position"] > queue.loc[
        queue["has_position_data"], "avg_position"
    ].median():
        return "RANKING_SLIP"

    if (
        row["impressions"] >= queue["impressions"].quantile(0.75)
        and row["ctr"] <= queue["ctr"].quantile(0.25)
    ):
        return "HIGH_IMPRESSIONS_LOW_CTR"

    if row["impressions"] >= queue["impressions"].median():
        return "CONTENT_OPPORTUNITY"

    return "NO_STRONG_SIGNAL"

queue["reason_code"] = queue.apply(assign_reason, axis=1)

# Map reason → recommended human action
action_map = {
    "DECAY_RISK": "Review and refresh the page for current relevance and search intent.",
    "RANKING_SLIP": "Review content quality, topical coverage, and search intent alignment.",
    "HIGH_IMPRESSIONS_LOW_CTR": "Review title, snippet, intent alignment, and SERP presentation.",
    "CONTENT_OPPORTUNITY": "Assess whether targeted content expansion would add useful coverage.",
    "NO_STRONG_SIGNAL": "Defer unless business or editorial context gives another reason to review.",
}

queue["recommended_action"] = queue["reason_code"].map(action_map)

# Rank highest priority first
queue = queue.sort_values(
    ["priority_score", "impressions"],
    ascending=[False, False]
).reset_index(drop=True)

queue["priority_rank"] = np.arange(1, len(queue) + 1)

# Keep the paper-facing queue compact.
output_columns = [
    "priority_rank",
    "content_id",
    "client_id",
    "priority_score",
    "reason_code",
    "recommended_action",
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
]

output_columns = [c for c in output_columns if c in queue.columns]

action_queue = queue[output_columns].head(100).copy()

print("\nTop 10 recommended actions:")
display(action_queue.head(10))

print("\nReason-code distribution:")
display(action_queue["reason_code"].value_counts().to_frame("count"))

### Intended use

The playbook is intended for SEO and content teams that need to prioritize limited editorial review time. It converts model or observed search-performance signals into a ranked review queue.

The primary use is prioritization: deciding which content items deserve human attention first.

### Limits

The score is not a guarantee of future ranking improvement. The underlying data are observational and describe historical search-performance patterns. The playbook therefore supports prioritization rather than causal claims about the effect of refreshing content.

The queue should not be treated as a production publishing system. Recommendations should be interpreted together with search intent, content quality, business importance, factual accuracy, and editorial context.

The starter data are anonymized and represent a limited historical slice. Results may not generalize to other sites, industries, time periods, or search environments without further validation.

In [ ]:
# SECTION 2 — Basic scope and coverage checks

print("Number of content items in source data:", len(df))
print("Number of items in action queue:", len(action_queue))
print("Number of clients represented:", df["client_id"].nunique())

print("\nPriority-score summary:")
display(action_queue["priority_score"].describe())

print("\nMissingness in key review fields:")
display(
    action_queue[
        ["impressions", "clicks", "ctr", "avg_position"]
    ].isna().mean().to_frame("missing_rate")
)

## 4. Monitoring / retrain triggers

### Human-review rules

Every recommended action requires human review before implementation.

A reviewer should check:

1. Whether the page still matches the intended search intent.
2. Whether the information is accurate, current, and complete.
3. Whether the observed signal is large enough to justify editorial effort.
4. Whether the page has strategic or business importance.
5. Whether competitors provide materially better coverage.
6. Whether the proposed change could damage useful existing content.
7. Whether the recommendation is consistent with the site's editorial standards.

### No-go cases

The system should not automatically:

- publish or rewrite content;
- delete a page;
- change a canonical URL;
- create redirects;
- change factual, legal, medical, financial, or safety-critical claims;
- remove content solely because its score is low;
- infer search intent without human review;
- treat a model score as proof of causation;
- make irreversible SEO changes.

The model produces a review queue, not an autonomous content-management decision.

In [ ]:
# SECTION 3 — Human-review gate

NO_GO_ACTIONS = {
    "auto_publish",
    "auto_delete",
    "auto_redirect",
    "auto_canonical_change",
    "auto_rewrite_factual_claims",
}

# The queue itself only proposes review actions.
action_queue["requires_human_review"] = True
action_queue["automation_allowed"] = False

assert action_queue["requires_human_review"].all()
assert not action_queue["automation_allowed"].any()

print("Human-review gate: PASS")
print("Automatic publishing/deletion/redirect actions: NOT ALLOWED")


## 5. Exports for the paper
### Monitoring and retraining policy

The playbook should be monitored because search behavior, content inventories, and traffic patterns can change over time.

Monitoring should cover:

- the distribution of priority scores;
- the volume of high-priority recommendations;
- changes in impressions, CTR, and average position;
- the distribution of important input features;
- the proportion of recommendations that reviewers consider useful;
- validation performance when new labeled outcomes become available.

### Retraining triggers

Retraining should be investigated when there is sustained evidence that the model no longer represents the current data or that recommendation quality has deteriorated.

Potential triggers include:

- meaningful input-data drift;
- a sustained change in the score distribution;
- degradation in the validated ranking/classification metrics;
- systematic disagreement between recommendations and human review;
- a material change in the data-generating process.

A single unusual observation should trigger investigation rather than automatic retraining. Retraining should be performed only after checking data quality, label definitions, leakage, and evaluation design.

In [ ]:
# SECTION 4 — Lightweight monitoring checks

monitoring = {
    "queue_size": len(action_queue),
    "high_priority_count": int(
        (action_queue["priority_rank"] <= 20).sum()
    ),
    "mean_priority_score": float(
        action_queue["priority_score"].mean()
    ),
    "median_priority_score": float(
        action_queue["priority_score"].median()
    ),
    "missing_rate_avg_position": float(
        action_queue["avg_position"].isna().mean()
    ),
}

monitoring_df = pd.DataFrame(
    monitoring.items(),
    columns=["metric", "value"]
)

display(monitoring_df)

print(
    "\nMonitoring rule: investigate material distribution or validation changes "
    "before deciding whether retraining is necessary."
)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.